## Variáveis de ambiente (3) — Wyscout

`download_data.py` e `download_heatmaps.py` usam os mesmos três valores:

- `WYSCOUT_SEARCH_TOKEN` — token da API (query param `token`)
- `WYSCOUT_GROUP_ID` — ex.: `1432001`
- `WYSCOUT_SUBGROUP_ID` — ex.: `479792`

### No terminal (zsh/bash)

```bash
export WYSCOUT_SEARCH_TOKEN="…"
export WYSCOUT_GROUP_ID="…"
export WYSCOUT_SUBGROUP_ID="…"
```

### Neste notebook

Preenche os valores na célula seguinte **antes** de correr o download.

In [5]:
import os
from pathlib import Path


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "download_data.py").is_file():
            return p
    raise FileNotFoundError(
        "Não encontrei a raiz do repo (scripts/download_data.py). "
        "Abre o notebook a partir de raumdeuterapp ou faz cd para essa pasta."
    )


ROOT = find_repo_root()
os.chdir(ROOT)
print("Repo root:", ROOT)

# Preencher (ou já exportadas no shell antes de iniciar Jupyter)
os.environ["WYSCOUT_SEARCH_TOKEN"] = "34edd9fc7fc3bb05fd09472940371b53d043ef98"
os.environ["WYSCOUT_GROUP_ID"] = "1432001"
os.environ["WYSCOUT_SUBGROUP_ID"] = "479792"

Repo root: /Users/fbobiano/Projects/raumdeuterappv2


## 1. Download — `download_data.py`

Descarrega resultados da pesquisa Wyscout para `data/players/wyscout/` (um CSV por liga/temporada). Há ~2 s entre pedidos HTTP.

In [71]:
%run scripts/download_data.py

Fetching Premier League (competition=8, season=-6963 / 2025/2026) -> Premier League 25-26.csv
  page 0: got 500 new rows (+0 dups) (total so far: 500) | page_current=0 page_count=2 next=yes
  page 1 attempt 1: 6 dup(s) — retrying in 6s...
  page 1 attempt 2: 13 dup(s) — retrying in 6s...
  page 1: got 30 new rows (+0 dups) (total so far: 530) | page_current=1 page_count=2 next=yes
  wrote 530 rows
Fetching Premier League (competition=8, season=-5249 / 2024/2025) -> Premier League 24-25.csv
  page 0: got 500 new rows (+0 dups) (total so far: 500) | page_current=0 page_count=2 next=yes
  page 1 attempt 1: 36 dup(s) — retrying in 6s...
  page 1 attempt 2: 36 dup(s) — retrying in 6s...
  page 1: got 65 new rows (+0 dups) (total so far: 565) | page_current=1 page_count=2 next=yes
  wrote 565 rows
Fetching Premier League (competition=8, season=-4330 / 2023/2024) -> Premier League 23-24.csv
  page 0: got 500 new rows (+0 dups) (total so far: 500) | page_current=0 page_count=2 next=yes
  page 

## 1.b Heatmaps Wyscout — `download_heatmaps.py`

Opcional: **depois** de existirem os parquets consolidados `data/players/all/{ano}_all_leagues.parquet` (gerados mais abaixo no pipeline). Usa os **mesmos** três tokens que na secção inicial.

Escreve `data/players/heatmaps/heatmaps_2025.parquet` e `heatmaps_2026.parquet` (GraphQL `playerHeatmap`, 5 workers em paralelo; re-correr só preenche pares em falta).

### No terminal (operacional — precisa das variáveis Wyscout)

```bash
WYSCOUT_SEARCH_TOKEN=… WYSCOUT_GROUP_ID=… WYSCOUT_SUBGROUP_ID=… \
  apps/api/.venv/bin/python scripts/download_heatmaps.py
```

Substitui `…` pelos valores (ou usa `export` nas três linhas e um `apps/api/.venv/bin/python scripts/download_heatmaps.py` sem prefixo).

### A partir deste notebook

Corre a célula seguinte **depois** da célula que define `ROOT` e `os.environ["WYSCOUT_*"]`; usa o Python da venv `apps/api` (onde estão `pandas` / `pyarrow`).

In [6]:
import subprocess

venv_python = ROOT / "apps" / "api" / ".venv" / "bin" / "python"
script = ROOT / "scripts" / "download_heatmaps.py"
if not venv_python.is_file():
    raise FileNotFoundError(
        f"Falta {venv_python} — na raiz do repo: cd apps/api && uv sync"
    )
subprocess.run([str(venv_python), str(script)], check=True, cwd=ROOT)


=== season 2025 ===
  source: /Users/fbobiano/Projects/raumdeuterappv2/data/players/all/2025_all_leagues.parquet
  output: /Users/fbobiano/Projects/raumdeuterappv2/data/players/heatmaps/heatmaps_2025.parquet
  WARN: 8164 rows with unknown Competition (no competition_id) — skipping. Names: ['1 Lyga', '1. Deild', '1. Division', '1. HNL Juniori', '1. Lig', '1. Liga Classic', '1. Liga Promotion', '1. SNL', '1a Divisió', '1st Division', '2. Division', '2. Lig', '2. Liga', '2. Liga Interregional', '2nd Division', '3. Division', '3. Liga', 'A Lyga', 'A-League', 'A.LeCoq Premium Liiga', 'Abissnet Superiore', 'Allsvenskan U19', 'Arabian Gulf Reserve League', 'Baiano 1', 'Baiano U20', 'Besta-deild karla', 'Botola Pro', 'Brasileiro U17', 'Bölgesel Amatör Lig', 'CSL', 'Campeonato Nacional U18', 'Campeonato Nacional U20', 'Campionato Nazionale Under 17 A&B', 'Campionato Primavera 1', 'Campionato Primavera 2', 'Campionato Primavera 3 - Dante Berretti', 'Canadian Premier League', 'Carioca 1', 'Cario

CompletedProcess(args=['/Users/fbobiano/Projects/raumdeuterappv2/apps/api/.venv/bin/python', '/Users/fbobiano/Projects/raumdeuterappv2/scripts/download_heatmaps.py'], returncode=0)

## 2. Limpeza — `cleaning_data.py`

Lê/escreve UTF-8 em `data/players/wyscout/*.csv` e normaliza células que parecem listas Python (ex.: `['LWF', 'LW']` → texto separado por vírgulas).

In [73]:
%run scripts/cleaning_data.py

1. HNL 15-16.csv: 200 rows
1. HNL 16-17.csv: 197 rows
1. HNL 17-18.csv: 206 rows
1. HNL 18-19.csv: 319 rows
1. HNL 19-20.csv: 328 rows
1. HNL 20-21.csv: 344 rows
1. HNL 21-22.csv: 349 rows
1. HNL 22-23.csv: 350 rows
1. HNL 23-24.csv: 378 rows
1. HNL 24-25.csv: 334 rows
1. HNL 25-26.csv: 320 rows
2. Bundesliga 15-16.csv: 241 rows
2. Bundesliga 16-17.csv: 300 rows
2. Bundesliga 17-18.csv: 304 rows
2. Bundesliga 18-19.csv: 456 rows
2. Bundesliga 19-20.csv: 484 rows
2. Bundesliga 20-21.csv: 493 rows
2. Bundesliga 21-22.csv: 507 rows
2. Bundesliga 22-23.csv: 496 rows
2. Bundesliga 23-24.csv: 494 rows
2. Bundesliga 24-25.csv: 523 rows
2. Bundesliga 25-26.csv: 533 rows
Allsvenskan 2015.csv: 151 rows
Allsvenskan 2016.csv: 187 rows
Allsvenskan 2017.csv: 221 rows
Allsvenskan 2018.csv: 373 rows
Allsvenskan 2019.csv: 388 rows
Allsvenskan 2020.csv: 401 rows
Allsvenskan 2021.csv: 410 rows
Allsvenskan 2022.csv: 423 rows
Allsvenskan 2023.csv: 454 rows
Allsvenskan 2024.csv: 455 rows
Allsvenskan 2025.cs

## 2.5. Enriquecimento Transfermarkt — `enrich_with_tm.py`

Corre **depois** da limpeza (passo 2). Usa `data/tm/people.csv` e `data/tm/players.csv` para juntar `image_url` e altura (`height_in_cm` → coluna `Height`) aos CSV Wyscout em `data/players/wyscout/` (sobrescreve in-place).

- Primeira execução: `--dry-run` (preview; não grava).
- Segunda execução: sem `--dry-run` para gravar.

In [74]:
%run ./scripts/enrich_with_tm.py 

Loading TM mapping…
  141,183 distinct player ids (Wyscout-id + Soccerway-id) with TM rows
Enriching 450 wyscout CSVs…
  1. HNL 15-16.csv: rows=200 image_url_total=119 height_filled=200
  1. HNL 16-17.csv: rows=197 image_url_total=121 height_filled=197
  1. HNL 17-18.csv: rows=206 image_url_total=125 height_filled=206
  1. HNL 18-19.csv: rows=319 image_url_total=309 height_filled=319
  1. HNL 19-20.csv: rows=328 image_url_total=313 height_filled=328
  1. HNL 20-21.csv: rows=344 image_url_total=329 height_filled=344
  1. HNL 21-22.csv: rows=349 image_url_total=312 height_filled=349
  1. HNL 22-23.csv: rows=350 image_url_total=320 height_filled=350
  1. HNL 23-24.csv: rows=378 image_url_total=323 height_filled=378
  1. HNL 24-25.csv: rows=334 image_url_total=266 height_filled=334
  1. HNL 25-26.csv: rows=320 image_url_total=229 height_filled=320
  2. Bundesliga 15-16.csv: rows=241 image_url_total=190 height_filled=241
  2. Bundesliga 16-17.csv: rows=300 image_url_total=207 height_filled=

## 3. New performance Index — `new_performance_index.py`


In [75]:
%run ./transformation/new_performance_index.py

⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chil

## 4. CSV → Parquet — `csv_to_parquet.py`

Converte cada `*_all_leagues.csv` em `data/players/all/` para `.parquet` (snappy). Os CSV agregados por época precisam de já existir nessa pasta (se os geras noutro script, corre esse passo antes deste).

In [76]:
%run ./scripts/csv_to_parquet.py

  2015_all_leagues.csv -> 2015_all_leagues.parquet ... 19.9 MB -> 6.4 MB  (32%), CSV removed
  2016_all_leagues.csv -> 2016_all_leagues.parquet ... 26.6 MB -> 8.2 MB  (31%), CSV removed
  2017_all_leagues.csv -> 2017_all_leagues.parquet ... 28.7 MB -> 8.8 MB  (31%), CSV removed
  2018_all_leagues.csv -> 2018_all_leagues.parquet ... 43.0 MB -> 12.9 MB  (30%), CSV removed
  2019_all_leagues.csv -> 2019_all_leagues.parquet ... 43.6 MB -> 13.0 MB  (30%), CSV removed
  2020_all_leagues.csv -> 2020_all_leagues.parquet ... 47.3 MB -> 13.9 MB  (29%), CSV removed
  2021_all_leagues.csv -> 2021_all_leagues.parquet ... 51.4 MB -> 15.0 MB  (29%), CSV removed
  2022_all_leagues.csv -> 2022_all_leagues.parquet ... 50.7 MB -> 14.8 MB  (29%), CSV removed
  2023_all_leagues.csv -> 2023_all_leagues.parquet ... 53.0 MB -> 15.3 MB  (29%), CSV removed
  2024_all_leagues.csv -> 2024_all_leagues.parquet ... 49.2 MB -> 14.2 MB  (29%), CSV removed
  2025_all_leagues.csv -> 2025_all_leagues.parquet ... 53.8 MB 

## 5. Update potential scores— `train_potential.py`

In [ ]:
%run scripts/train_potential.py

## check duplicates

In [72]:
%run scripts/check_wyscout_duplicate_players.py --only-with-dups


file                                                        rows   dup_keys  excess_rows  notes
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
Files scanned: 450  With duplicate keys: 0  Sum of duplicate_key counts: 0


## Remover duplicados nos CSV Wyscout — `dedupe_wyscout_csv_rows.py`

Por ficheiro em `data/players/wyscout/`, remove linhas repetidas com a mesma chave **(Wyscout id, nome do jogador, equipa)**. Por omissão mantém a **última** ocorrência (`--keep last`); usa `--keep first` ou `--dry-run` no terminal conforme precises.

**Ordem lógica:** corre isto logo **após** a limpeza (secção 2) e **antes** do índice de performance (secção 3). Esta célula está no fim do notebook só como referência ao script; sobe-a ou corre-a no momento certo do pipeline.

In [46]:
%run scripts/dedupe_wyscout_csv_rows.py

2. Bundesliga 21-22.csv: 512 -> 507 rows (removed 5)
2. Bundesliga 24-25.csv: 523 -> 515 rows (removed 8)
Argentina LPF 20-21.csv: 654 -> 652 rows (removed 2)
Argentina LPF 2021.csv: 933 -> 929 rows (removed 4)
Argentina LPF 2022.csv: 947 -> 945 rows (removed 2)
Argentina LPF 2023.csv: 966 -> 963 rows (removed 3)
Argentina LPF 2025.csv: 1021 -> 1016 rows (removed 5)
Argentina LPF 2026.csv: 824 -> 808 rows (removed 16)
Belgian Pro League 18-19.csv: 516 -> 503 rows (removed 13)
Belgian Pro League 20-21.csv: 532 -> 515 rows (removed 17)
Brasileirão 2018.csv: 697 -> 687 rows (removed 10)
Brasileirão 2020.csv: 724 -> 723 rows (removed 1)
Campeonato de Portugal 25-26.csv: 1616 -> 1582 rows (removed 34)
Championship 19-20.csv: 676 -> 674 rows (removed 2)
Championship 23-24.csv: 706 -> 697 rows (removed 9)
Championship 24-25.csv: 748 -> 745 rows (removed 3)
Chilean Primera Division 2021.csv: 537 -> 526 rows (removed 11)
Ekstraklasa 22-23.csv: 522 -> 508 rows (removed 14)
Ekstraklasa 23-24.csv:

## 10. Filtered player valuations — `build_player_valuations.py`

Cria `data/tm/player_valuations_filtered.csv` (e parquet com `--parquet`) com colunas
`wyscout_id`, `key_transfermarkt`, `date`, `market_value_in_eur`. Por defeito mantém
qualquer jogador que apareça em pelo menos um ficheiro Wyscout (`--mode union`).

_Para uso estrito (jogadores em **todos** os ficheiros — geralmente devolve 0):
`--mode intersection`._


In [13]:
%run build_player_valuations.py --parquet

Scanning Wyscout files for player ids…
  440 files scanned
  44,715 export player ids in union
  21,411 people rows mapped to key_transfermarkt (wyscout + soccerway)
Reading player_valuations.csv…
  317,864 valuation rows after player_id filter
Wrote /Users/fbobiano/Projects/raumdeuterappv2/data/tm/player_valuations_filtered.csv (317,864 rows)
Wrote /Users/fbobiano/Projects/raumdeuterappv2/data/tm/player_valuations_filtered.parquet


## 11. Club logos parquet — `build_club_logos.py`

Percorre todos os CSVs Wyscout e produz `data/teams/club_logos.parquet` com colunas `team`, `logo_url`, `n_seasons`, `latest_file`. Inclui todas as equipas (sem filtro).

In [ ]:
%run build_club_logos.py